In [ ]:
%load_ext autoreload
%autoreload 2

# Tutorial 7. DistributedAnalysis over PyAEDT (gRPC) — no COM
##### The same EPR workflow, with one flag: `use_pyaedt=True`
Author: Joey Yaker

## Introduction

pyEPR drives HFSS through a hand-rolled **COM** layer (`pyEPR.ansys`), which is Windows-only and tied to a private scripting interface. By passing **`use_pyaedt=True`** to `ProjectInfo`, pyEPR's *existing* `DistributedAnalysis` connects through **PyAEDT** (`ansys-aedt-core`) — Ansys's official, maintained Python API — **entirely over gRPC, with no COM**.

Nothing else about the workflow changes. Junctions, modes, `do_EPR_analysis()`, `QuantumAnalysis` — all identical. The COM path is byte-for-byte unchanged when `use_pyaedt=False` (the default), and the physics matches the COM path **digit-for-digit** (`p_mj = 0.9755` on this demo transmon).

Under the hood only two things differ — the connection (`win32com.Dispatch` vs `ansys.aedt.core.Hfss` → `hfss.odesktop`) and the field-calculator read-back inside `CalcObject.evaluate` (`ClcEval` + `GetTopEntryValue` vs `CalculatorWrite` to a `.fld` file). The field-calculator stack and the `EditSources` normalization are identical and run fine over gRPC.

## <div style="background:#BBFABB;line-height:2em;">Requirements<div>

PyAEDT is an optional extra, imported lazily — only when `use_pyaedt=True` — so `import pyEPR` never requires it:

```bash
pip install "pyEPR-quantum[pyaedt]"
```

You also need a **solved** HFSS eigenmode design.

## <div style="background:#BBFABB;line-height:2em;">Describe the project — with use_pyaedt=True<div>

This is the ordinary `ProjectInfo`, plus `use_pyaedt=True` (and the AEDT version). `do_connect=False` here just defers the connection to the next cell so the junction can be declared first.

In [1]:
import numpy as np
import pyEPR as epr

pinfo = epr.ProjectInfo(
    project_path=r"D:\JYaker\HFSS_Projects",
    project_name="EPR_Demo_Project",
    design_name="EPR_Sample_Demo",
    setup_name="EPR_Scan",
    do_connect=False,
    use_pyaedt=True,            # <-- connect over PyAEDT / gRPC instead of COM
    aedt_version="2026.1",
)
pinfo.junctions["j1"] = {"Lj_variable": "Lj_Transmon", "line": "Junction_line"}

## <div style="background:#BBFABB;line-height:2em;">Connect and extract — pyEPR's normal pipeline, over gRPC<div>

`pinfo.connect()` opens/attaches over gRPC. From here it is the standard pyEPR flow: `DistributedAnalysis.do_EPR_analysis()` runs the field extraction (every `CalcObject.evaluate` reads back with `CalculatorWrite`), and prints the junction participation per mode.

In [2]:
pinfo.connect()                                   # gRPC connection
print("connected over gRPC:", pinfo.design._is_grpc)

eprd = epr.DistributedAnalysis(pinfo)
eprd.do_EPR_analysis()                            # pyEPR's pipeline, over gRPC

connected over gRPC: True
    Calculating junction EPR, method=`line_voltage`
        junction   EPR p_1j   sign s_1j    (p_capacitive)
        j1         0.975461   (-)          0.0230754


The qubit-mode participation is **`p_mj = 0.975461`** — identical to what pyEPR's COM path extracts from the same solved design. Results are saved to `eprd.data_filename`; load them with `QuantumAnalysis` exactly as in Tutorial 1 to get the dressed frequencies and the chi / anharmonicity matrix.

In [3]:
epra = epr.QuantumAnalysis(eprd.data_filename)
epra.analyze_all_variations()
# ... same QuantumAnalysis API as the COM workflow (Tutorial 1)

## Same physics, no COM

The participation `p_mj = 0.9755` — and every number downstream of it — is **identical** to pyEPR's COM path, digit-for-digit. The only thing that changed is one keyword: `use_pyaedt=True`, which routes the *same* `DistributedAnalysis` over Ansys's maintained PyAEDT gRPC API instead of COM.

* **No COM / `pywin32`** — gRPC, the default for AEDT since 2022 R2.
* **Attaches to the session that owns the project** (via its `.aedt.lock`), avoiding stale-session and *project-locked* errors.
* **Maintenance moves to Ansys** — `ansys-aedt-core` is officially versioned and supported.

For COM users nothing changes: with `use_pyaedt=False` (the default) the COM path is byte-for-byte unchanged, and PyAEDT stays an optional extra.